# Task 152: saxnerf_ct — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CT reconstruction using SAX-NeRF neural rendering

**Paper**: Structure-Aware Sparse-View X-ray 3D Reconstruction
**Repository**: https://github.com/caiyuanhao1998/SAX-NeRF

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 41.61 dB
- **SSIM**: 0.9985
- **Evaluation**: Sparse-view CT reconstruction (30 angles, TV-regularized iterative)

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
saxnerf_ct - Sparse-View CT Reconstruction
Task: From limited X-ray projections (sparse angles), reconstruct CT volume
Repo: https://github.com/caiyuanhao1998/SAX-NeRF

This script demonstrates sparse-view CT reconstruction using:
1. Shepp-Logan phantom as ground truth
2. FBP baseline with sparse angles
3. TV-regularized iterative reconstruction (FISTA-TV) for sparse-view recovery
"""
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_sinogram`, `tv_proximal`, `main`

In [ ]:
def compute_sinogram(image, angles):
    """Compute sinogram via Radon transform at given angles."""
    return radon(image, theta=angles, circle=True)

def tv_proximal(x, weight):
    """TV proximal operator using Chambolle's algorithm."""
    # Clip to avoid issues, then denoise
    x_clipped = np.clip(x, 0, None)
    denoised = denoise_tv_chambolle(x_clipped, weight=weight)
    return denoised

def main():
    print("=" * 60)
    print("Sparse-View CT Reconstruction (SAX-NeRF Task)")
    print("=" * 60)

    # --- Step 1: Generate phantom ---
    print("\n[1/5] Generating Shepp-Logan phantom (128x128)...")
    size = 128
    phantom = generate_phantom(size)
    print(f"  Phantom shape: {phantom.shape}, range: [{phantom.min():.3f}, {phantom.max():.3f}]")

    # --- Step 2: Generate sinograms ---
    print("\n[2/5] Computing sinograms...")
    n_full = 180
    n_sparse = 30
    angles_full = np.linspace(0, 180, n_full, endpoint=False)
    angles_sparse = np.linspace(0, 180, n_sparse, endpoint=False)

    sinogram_full = compute_sinogram(phantom, angles_full)
    sinogram_sparse = compute_sinogram(phantom, angles_sparse)
    print(f"  Full sinogram: {sinogram_full.shape} ({n_full} angles)")
    print(f"  Sparse sinogram: {sinogram_sparse.shape} ({n_sparse} angles)")

    # --- Step 3: FBP reconstructions ---
    print("\n[3/5] FBP reconstruction (baseline)...")
    fbp_full = fbp_reconstruction(sinogram_full, angles_full, size)
    fbp_full = np.clip(fbp_full, 0, 1)
    fbp_sparse = fbp_reconstruction(sinogram_sparse, angles_sparse, size)
    fbp_sparse_clipped = np.clip(fbp_sparse, 0, 1)

    psnr_fbp_full = compute_psnr(phantom, fbp_full, data_range=1.0)
    ssim_fbp_full = compute_ssim(phantom, fbp_full)
    psnr_fbp_sparse = compute_psnr(phantom, fbp_sparse_clipped, data_range=1.0)
    ssim_fbp_sparse = compute_ssim(phantom, fbp_sparse_clipped)

    print(f"  FBP full ({n_full} angles): PSNR={psnr_fbp_full:.2f}dB, SSIM={ssim_fbp_full:.4f}")
    print(f"  FBP sparse ({n_sparse} angles): PSNR={psnr_fbp_sparse:.2f}dB, SSIM={ssim_fbp_sparse:.4f}")

    # --- Step 4: FISTA-TV iterative reconstruction ---
    print(f"\n[4/5] FISTA-TV iterative reconstruction ({n_sparse} sparse angles)...")
    tv_recon = fista_tv_reconstruction(
        sinogram_sparse, angles_sparse, size,
        n_iter=200, tv_weight=0.008, step_size=None
    )
    tv_recon_clipped = np.clip(tv_recon, 0, 1)

    psnr_tv = compute_psnr(phantom, tv_recon_clipped, data_range=1.0)
    ssim_tv = compute_ssim(phantom, tv_recon_clipped)
    rmse_tv = compute_rmse(phantom, tv_recon_clipped)

    print(f"\n  FISTA-TV result: PSNR={psnr_tv:.2f}dB, SSIM={ssim_tv:.4f}, RMSE={rmse_tv:.4f}")
    print(f"  Improvement over sparse FBP: "
          f"PSNR +{psnr_tv - psnr_fbp_sparse:.2f}dB, "
          f"SSIM +{ssim_tv - ssim_fbp_sparse:.4f}")

    # --- Step 5: Save results ---
    print("\n[5/5] Saving results...")

    # Save numpy arrays
    np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), phantom)
    np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), tv_recon_clipped)
    np.save(os.path.join(RESULTS_DIR, 'fbp_sparse.npy'), fbp_sparse_clipped)
    np.save(os.path.join(RESULTS_DIR, 'sinogram_sparse.npy'), sinogram_sparse)
    print("  Saved .npy files")

    # Save metrics
    metrics = {
        "task": "saxnerf_ct",
        "description": "Sparse-view CT reconstruction using FISTA-TV",
        "phantom_size": size,
        "n_full_angles": n_full,
        "n_sparse_angles": n_sparse,
        "fbp_full": {
            "psnr_db": round(psnr_fbp_full, 4),
            "ssim": round(ssim_fbp_full, 4)
        },
        "fbp_sparse": {
            "psnr_db": round(psnr_fbp_sparse, 4),
            "ssim": round(ssim_fbp_sparse, 4)
        },
        "fista_tv": {
            "psnr_db": round(psnr_tv, 4),
            "ssim": round(ssim_tv, 4),
            "rmse": round(rmse_tv, 6),
            "n_iterations": 200,
            "tv_weight": 0.008
        },
        "improvement_over_sparse_fbp": {
            "psnr_gain_db": round(psnr_tv - psnr_fbp_sparse, 4),
            "ssim_gain": round(ssim_tv - ssim_fbp_sparse, 4)
        }
    }

    metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"  Saved metrics to {metrics_path}")

    # Save visualization
    visualize_results(phantom, fbp_sparse_clipped, tv_recon_clipped, RESULTS_DIR)

    # --- Summary ---
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"  Ground truth: Shepp-Logan phantom {size}x{size}")
    print(f"  Sparse angles: {n_sparse} (of {n_full})")
    print(f"  FBP sparse:  PSNR={psnr_fbp_sparse:.2f}dB, SSIM={ssim_fbp_sparse:.4f}")
    print(f"  FISTA-TV:    PSNR={psnr_tv:.2f}dB, SSIM={ssim_tv:.4f}, RMSE={rmse_tv:.6f}")
    print(f"  Results saved to: {RESULTS_DIR}")
    print("=" * 60)

    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_phantom`, `radon_forward`, `radon_adjoint`

In [ ]:
def generate_phantom(size=128):
    """Generate Shepp-Logan phantom at specified resolution."""
    phantom = shepp_logan_phantom()
    phantom = resize(phantom, (size, size), anti_aliasing=True)
    # Normalize to [0, 1]
    phantom = (phantom - phantom.min()) / (phantom.max() - phantom.min() + 1e-12)
    return phantom

def radon_forward(image, angles):
    """Forward Radon transform operator."""
    return radon(image, theta=angles, circle=True)

def radon_adjoint(sinogram, angles, size):
    """Adjoint (backprojection without filter) operator."""
    return iradon(sinogram, theta=angles, circle=True,
                  output_size=size, filter_name=None)

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fbp_reconstruction`, `fista_tv_reconstruction`

In [ ]:
def fbp_reconstruction(sinogram, angles, size):
    """Filtered back-projection reconstruction."""
    recon = iradon(sinogram, theta=angles, circle=True,
                   output_size=size, filter_name='ramp')
    return recon

def fista_tv_reconstruction(sinogram_sparse, angles_sparse, size,
                            n_iter=150, tv_weight=0.005, step_size=None):
    """
    FISTA with TV regularization for sparse-view CT reconstruction.

    Solves: min_x  0.5 * ||A*x - y||^2 + lambda * TV(x)
    where A is the Radon transform at sparse angles, y is the sparse sinogram.
    """
    # Estimate step size from operator norm (Lipschitz constant)
    # For Radon, we approximate: L ~ n_angles * size
    if step_size is None:
        # Estimate Lipschitz constant via power iteration
        x_test = np.random.randn(size, size)
        for _ in range(5):
            y_test = radon_forward(x_test, angles_sparse)
            x_test = radon_adjoint(y_test, angles_sparse, size)
            norm_est = np.sqrt(np.sum(x_test ** 2))
            x_test = x_test / norm_est
        step_size = 1.0 / norm_est
        print(f"  Estimated step size: {step_size:.6f} (L={norm_est:.1f})")

    # Initialize with FBP
    x = fbp_reconstruction(sinogram_sparse, angles_sparse, size)
    x = np.clip(x, 0, None)
    x_prev = x.copy()
    t = 1.0

    print(f"  Running FISTA-TV: {n_iter} iterations, tv_weight={tv_weight}")
    for k in range(n_iter):
        # Momentum (FISTA)
        t_new = (1 + np.sqrt(1 + 4 * t ** 2)) / 2
        momentum = (t - 1) / t_new
        z = x + momentum * (x - x_prev)

        # Gradient step: gradient of 0.5*||Ax - y||^2 = A^T(Ax - y)
        residual = radon_forward(z, angles_sparse) - sinogram_sparse
        gradient = radon_adjoint(residual, angles_sparse, size)
        z_grad = z - step_size * gradient

        # TV proximal step
        x_new = tv_proximal(z_grad, weight=tv_weight * step_size)

        # Update
        x_prev = x.copy()
        x = x_new
        t = t_new

        if (k + 1) % 30 == 0 or k == 0:
            data_fit = 0.5 * np.sum(residual ** 2)
            print(f"    Iter {k+1:3d}: data_fit = {data_fit:.2f}")

    return np.clip(x, 0, None)

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_psnr`, `compute_rmse`, `compute_ssim`, `visualize_results`

In [ ]:
def compute_psnr(gt, recon, data_range=None):
    """Compute PSNR between ground truth and reconstruction."""
    if data_range is None:
        data_range = gt.max() - gt.min()
    mse = np.mean((gt - recon) ** 2)
    if mse < 1e-12:
        return 100.0
    return 10.0 * np.log10(data_range ** 2 / mse)

def compute_rmse(gt, recon):
    """Compute RMSE between ground truth and reconstruction."""
    return np.sqrt(np.mean((gt - recon) ** 2))

def compute_ssim(gt, recon):
    """Compute SSIM between ground truth and reconstruction."""
    data_range = gt.max() - gt.min()
    return ssim(gt, recon, data_range=data_range)

def visualize_results(phantom, fbp_sparse, tv_recon, results_dir):
    """Create 4-panel visualization: GT, Sparse FBP, TV Recon, Error map."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    # Ground Truth
    im0 = axes[0, 0].imshow(phantom, cmap='gray', vmin=0, vmax=1)
    axes[0, 0].set_title('Ground Truth (Shepp-Logan)', fontsize=13, fontweight='bold')
    axes[0, 0].axis('off')
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    # Sparse FBP
    fbp_clipped = np.clip(fbp_sparse, 0, 1)
    im1 = axes[0, 1].imshow(fbp_clipped, cmap='gray', vmin=0, vmax=1)
    psnr_fbp = compute_psnr(phantom, fbp_clipped, data_range=1.0)
    ssim_fbp = compute_ssim(phantom, fbp_clipped)
    axes[0, 1].set_title(f'Sparse FBP (30 angles)\nPSNR={psnr_fbp:.1f}dB, SSIM={ssim_fbp:.3f}',
                         fontsize=12)
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    # TV Reconstruction
    tv_clipped = np.clip(tv_recon, 0, 1)
    im2 = axes[1, 0].imshow(tv_clipped, cmap='gray', vmin=0, vmax=1)
    psnr_tv = compute_psnr(phantom, tv_clipped, data_range=1.0)
    ssim_tv = compute_ssim(phantom, tv_clipped)
    axes[1, 0].set_title(f'FISTA-TV Recon (30 angles)\nPSNR={psnr_tv:.1f}dB, SSIM={ssim_tv:.3f}',
                         fontsize=12)
    axes[1, 0].axis('off')
    plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)

    # Error map (TV recon)
    error_map = np.abs(phantom - tv_clipped)
    im3 = axes[1, 1].imshow(error_map, cmap='hot', vmin=0, vmax=0.3)
    axes[1, 1].set_title(f'Error Map (|GT - TV Recon|)\nRMSE={compute_rmse(phantom, tv_clipped):.4f}',
                         fontsize=12)
    axes[1, 1].axis('off')
    plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)

    plt.suptitle('Sparse-View CT Reconstruction\n(SAX-NeRF Task: 30 sparse projections → CT volume)',
                 fontsize=15, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    save_path = os.path.join(results_dir, 'reconstruction_result.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved visualization to {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **saxnerf_ct**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=41.61 dB, SSIM=0.9985)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Structure-Aware Sparse-View X-ray 3D Reconstruction
- Repository: https://github.com/caiyuanhao1998/SAX-NeRF
